In [28]:
# Locate all FS-3 parquet files
from pathlib import Path
import polars as pl
import numpy as np
import sys
repo = Path.home() / "fs-data"
fs3_folder = repo / "FS-3"
files = sorted(fs3_folder.rglob("*.parquet"))
if not fs3_folder.is_dir():
    raise FileNotFoundError(fs3_folder)
print("Total FS-3 parquet files:", len(files))


Total FS-3 parquet files: 142


In [29]:
# Find AnalysisFunctions.py also Import simpleTimeCol()
locations = [
    repo,
    repo / "notebooks",
    Path.cwd()
]
for folder in locations:
    if (folder / "AnalysisFunctions.py").is_file():
        sys.path.insert(0, str(folder))
        print("Found AnalysisFunctions.py:", folder)
        break
else:
    raise FileNotFoundError("AnalysisFunctions.py not found")
from AnalysisFunctions import simpleTimeCol
print("Successfully imported simpleTimeCol!")

Found AnalysisFunctions.py: /home/JacksonLoewen/fs-data
Successfully imported simpleTimeCol!


In [30]:
#Adds missing timestamps and then save corrected copies cuz for some reason when I did it before it said like 72 were errors so I had to fix blank errors and the others that had missing functions.
corrected_folder = Path.home() / "FS-3-time-ms-corrected"
corrected_folder.mkdir(parents=True, exist_ok=True)
added = 0
preserved = 0
converted = 0
failed = []
for file in files:
    relative_path = file.relative_to(fs3_folder)
    output_file = corrected_folder / relative_path
    try:
        df = pl.read_parquet(file)
        if "Time_ms" in df.columns:
            updated_df = df
            preserved += 1
            if df.schema["Time_ms"] != pl.Float64:
                updated_df = updated_df.with_columns(
                    pl.col("Time_ms").cast(pl.Float64)
                )
                converted += 1
        else:
            time_ms = (
                simpleTimeCol(df) * 1000
            ).cast(pl.Float64).alias("Time_ms")
            if len(time_ms) != df.height:
                raise ValueError("Time length mismatch")
            updated_df = df.with_columns(time_ms)
            added += 1
        if updated_df.height != df.height:
            raise ValueError("Row count changed")
        output_file.parent.mkdir(
            parents=True, exist_ok=True
        )
        updated_df.write_parquet(output_file)
    except Exception as error:
        failed.append((str(relative_path), str(error)))
        print("FAILED:", relative_path, error)
print("\nPROCESSING COMPLETE")
print("Time_ms added:", added)
print("Existing timestamps preserved:", preserved)
print("Data types converted:", converted)
print("Failed:", len(failed))


PROCESSING COMPLETE
Time_ms added: 69
Existing timestamps preserved: 73
Data types converted: 2
Failed: 0


In [26]:
#Verifies the data it checks missing columns, checks for incorrect types and other errors occurred
passed = 0
failed_checks = []

for file in files:
    relative = file.relative_to(fs3_folder)
    processed = corrected_folder / relative

    try:
        original = pl.read_parquet(file)
        updated = pl.read_parquet(processed)

        assert original.height == updated.height, "Row count changed"
        assert "Time_ms" in updated.columns, "Missing Time_ms"
        assert updated.schema["Time_ms"] == pl.Float64, "Wrong type"
        assert all(
            col in updated.columns for col in original.columns
        ), "Missing original column"

        if "Time_ms" in original.columns:
            original_time = original["Time_ms"].cast(pl.Float64)

            assert original_time.equals(
                updated["Time_ms"]
            ), "Existing timestamps changed"

        else:
            time = updated["Time_ms"]

            assert time.null_count() == 0, "Null timestamps"
            assert time.is_sorted(), "Time not sorted"

            if len(time) > 0:
                assert np.isclose(time[0], 0), "Time not starting at zero"

        passed += 1

    except Exception as error:
        failed_checks.append((str(relative), str(error)))

print("FINAL VERIFICATION")
print("Passed:", passed)
print("Failed:", len(failed_checks))

for name, error in failed_checks:
    print("FAILED:", name, error)


FINAL VERIFICATION
Passed: 142
Failed: 0


In [32]:
#Basically shows the results and it displays a sample of the complete data
processed_files = sorted(corrected_folder.rglob("*.parquet"))
print("Original files:", len(files))
print("Processed files:", len(processed_files))
print("All files processed:", len(files) == len(processed_files))
if missing_time := [
    f for f in files
    if "Time_ms" not in pl.read_parquet_schema(f)
]:
    sample = corrected_folder / missing_time[0].relative_to(fs3_folder)
    sample_df = pl.read_parquet(sample)
    print("\nExample:", sample.name)
    print("Time_ms type:", sample_df.schema["Time_ms"])
    print(sample_df.select("Time_ms").head(10))


Original files: 142
Processed files: 142
All files processed: True

Example: 08102025CoupleMoreFastWarmupsPostMechFixes.parquet
Time_ms type: Float64
shape: (10, 1)
┌────────────┐
│ Time_ms    │
│ ---        │
│ f64        │
╞════════════╡
│ 0.0        │
│ 11.916584  │
│ 23.833168  │
│ 35.749752  │
│ 47.666336  │
│ 59.58292   │
│ 71.499503  │
│ 83.416087  │
│ 95.332671  │
│ 107.249255 │
└────────────┘
